# Advanced Retrieval Enhancement

Transition the baseline RAG system into a professional, two-stage retrieval pipeline. This stage implements architectural enhancements to improve both recall and precision for municipal code queries.

### Objectives
1. **BM25 Keyword Indexing**: Implement a statistical keyword index to handle exact citations and legal identifiers.
2. **Hybrid Search (Fusion)**: Utilize Reciprocal Rank Fusion (RRF) to merge neural and keyword retrieval results.
3. **Neural Reranking**: Integrate a high-precision Cross-Encoder model to validate and re-sort top candidates.
4. **Comparative Analysis**: Benchmark the enhanced pipeline against the Stage 4 baseline and export evaluation logs.

## 1. Setup

Import necessary libraries for hybrid search, neural reranking, and rank fusion. Define paths for the processed chunks, the vector database, and the evaluation output.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
import re
from tqdm.auto import tqdm

# Input and Output Paths
DATA_PATH = Path('../data/processed/san_diego_code_chunks.jsonl')
DB_PATH = Path('../data/vector_store/san_diego_code_baseline')
EVAL_PATH = Path('../data/vector_store/enhanced_eval.csv')

# Load baseline data for keyword indexing
df = pd.read_json(DATA_PATH, lines=True)

# Initialize Neural Models
bi_encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# Initialize ChromaDB Persistent Client
chroma_client = chromadb.PersistentClient(DB_PATH)
collection = chroma_client.get_collection("san_diego_code")

print(f"Setup complete. Loaded {len(df)} records and initialized models.")

/Users/Aresh/Desktop/AAI 590/sd-land-use-rag/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/Users/Aresh/Desktop/AAI 590/sd-land-use-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup complete. Loaded 52384 records and initialized models.


## 2. Statistical Keyword Indexing

### 2.1 BM25 Tokenization

Define a specialized tokenizer to isolate legal identifiers and section numbers. Legal citations require preservation of numeric formatting which standard space-based tokenizers often fragment.

In [2]:
def tokenize_legal_text(text):
    """Extract alphanumeric identifiers and lowercase words for statistical indexing."""
    return re.findall(r'\w+', text.lower())

print("Tokenizing corpus for BM25 indexing...")
tokenized_corpus = [tokenize_legal_text(doc) for doc in tqdm(df['text'])]

Tokenizing corpus for BM25 indexing...


100%|██████████| 52384/52384 [00:00<00:00, 199412.08it/s]


### 2.2 BM25 Index Initialization

Initialize the BM25Okapi index to enable rapid, exact token matching across the 52k+ document chunks.

In [3]:
bm25 = BM25Okapi(tokenized_corpus)
print(f"BM25 Index initialized for {len(tokenized_corpus)} segments.")

BM25 Index initialized for 52384 segments.


## 3. Hybrid Search (Stage 1 Retrieval)

Implement **Reciprocal Rank Fusion (RRF)** to merge retrieval candidates from the neural (semantic) and BM25 (keyword) experts. This stage ensures high recall by capturing both conceptual meaning and exact regulatory identifiers.

In [4]:
def hybrid_search(query, n_results=10, rrf_k=60):
    """
    Merge results from Semantic and Keyword experts using Reciprocal Rank Fusion.
    """
    # 1. Neural Semantic Retrieval
    query_emb = bi_encoder.encode(query).tolist()
    neural_results = collection.query(query_embeddings=[query_emb], n_results=n_results*2)
    neural_ids = neural_results['ids'][0]
    
    # 2. Statistical Keyword Retrieval
    token_query = tokenize_legal_text(query)
    bm25_scores = bm25.get_scores(token_query)
    top_bm25_indices = np.argsort(bm25_scores)[-n_results*2:][::-1]
    bm25_ids = [f"chunk_{idx}" for idx in top_bm25_indices]
    
    # 3. Reciprocal Rank Fusion Implementation
    fused_scores = {}
    
    for rank, cid in enumerate(neural_ids):
        fused_scores[cid] = fused_scores.get(cid, 0) + 1 / (rank + rrf_k)
        
    for rank, cid in enumerate(bm25_ids):
        fused_scores[cid] = fused_scores.get(cid, 0) + 1 / (rank + rrf_k)
        
    # Sort by fused ranking score
    fused_sorted = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)[:n_results]
    fused_ids = [cid for cid, score in fused_sorted]
    
    return collection.get(ids=fused_ids, include=['documents', 'metadatas'])

## 4. Neural Reranking (Stage 2 Precision)

Utilize a **Cross-Encoder Reranker** to evaluate the specific semantic relatedness of the hybrid retrieval candidates. Unlike Bi-Encoders, Cross-Encoders process the query and document simultaneously to produce a highly accurate relevance score.

In [5]:
def advanced_query(query, n_top=5):
    """
    Execute two-stage pipeline: Hybrid Retrieval followed by Neural Reranking.
    """
    # Stage 1: Hybrid Retrieval (Recall)
    candidates = hybrid_search(query, n_results=25)
    docs = candidates['documents']
    metas = candidates['metadatas']
    ids = candidates['ids']
    
    # Stage 2: Cross-Encoder Reranking (Precision)
    # Prepare (Query, Document) pairs for the model
    pairs = [[query, doc] for doc in docs]
    ce_scores = cross_encoder.predict(pairs)
    
    # Re-sort results based on Cross-Encoder validation
    reranked_indices = np.argsort(ce_scores)[::-1]
    
    results = []
    for idx in reranked_indices[:n_top]:
        results.append({
            'id': ids[idx],
            'text': docs[idx],
            'metadata': metas[idx],
            'ce_relevance_score': float(ce_scores[idx])
        })
        
    return results

print("Reranking pipeline initialized.")

Reranking pipeline initialized.


## 5. Benchmarking & Comparative Evaluation

### 5.1 Pipeline Spot-Check

Test the enhanced pipeline on a complex query involving specific section identifiers.

In [6]:
test_q = "How is building height measured according to section 113.0201?"
test_res = advanced_query(test_q, n_top=3)

print(f"\n--- Query: '{test_q}' ---")
for i, r in enumerate(test_res):
    print(f"[Result {i+1} | CE Score: {r['ce_relevance_score']:.4f}]")
    print(f"Source: {r['metadata'].get('section', 'Unknown')} ({r['metadata'].get('filename')})")
    print(f"Snippet: {r['text'][:200]}...\n")


--- Query: 'How is building height measured according to section 113.0201?' ---
[Result 1 | CE Score: 7.0400]
Source: Structure Height (Ch11Art03Division02.pdf)
Snippet: Article 3: Land Development Procedures > Structure Height: Plumb line measurement. The structure height is measured from all points on top of a structure to existing grade or proposed grade, whichever...

[Result 2 | CE Score: 6.4354]
Source: Structure Height of Buildings and Structures (Excluding Fences, Retaining Walls, or Signs) (Ch11Art03Division02.pdf)
Snippet: Article 3: Land Development Procedures > Structure Height of Buildings and Structures (Excluding Fences, Retaining Walls, or Signs): The maximum permitted structure height is specified in the applicab...

[Result 3 | CE Score: 4.4084]
Source: (ii) (Ch11Art03Division02.pdf)
Snippet: Article 3: Land Development Procedures > (ii): Exterior Subterranean Areas. The overall structure height measurement shall not include subterranean vehicular access, exterior su

### 5.2 Export Evaluation Logs

Execute standard benchmark queries and save the results to `enhanced_eval.csv` for downstream comparison with the baseline.

In [7]:
benchmark_queries = [
    "What are the height limits for residential zones?",
    "What is the definition of a front yard setback?",
    "Where is multiple dwelling unit development permitted?",
    "What are the parking requirements for commercial zones?",
    "How is building height measured section 113.0201?"
]

eval_logs = []
for q in tqdm(benchmark_queries):
    results = advanced_query(q, n_top=5)
    for i, item in enumerate(results):
        eval_logs.append({
            'Query': q,
            'Rank': i+1,
            'Rerank_Score': item['ce_relevance_score'],
            'Source': f"{item['metadata'].get('section', 'Unknown')} ({item['metadata'].get('filename')})",
            'Text_Snippet': item['text'][:300]
        })

pd.DataFrame(eval_logs).to_csv(EVAL_PATH, index=False)
print(f"Enhanced evaluation logs exported to {EVAL_PATH}")

100%|██████████| 5/5 [00:00<00:00,  8.19it/s]

Enhanced evaluation logs exported to ../data/vector_store/enhanced_eval.csv
